# Sequential time-series regression — exploration & baseline

A demonstration report over the committed reference run in `OUTPUTS/example/`.

**The model.** A Conv1D → stacked Bi-GRU → linear head regressor (Keras 3 on
TensorFlow), trained sequentially over `TRAIN/*.csv` and evaluated against every
`TEST/*.csv` after each stage.

**What this notebook does.** It *reads* a trained session — nothing is trained
here. Everything below comes from `OUTPUTS/example/`, a committed artifact
directory, so this report is deterministic and needs no data, no GPU and no
training run to reproduce. All logic lives in the tested `mlpp` package
(`apps/backend/src/mlpp`); the cells only call into it.

To train your own session or score your own CSV, see the two CLIs at the end.

## The session contract

Every session directory carries a `manifest.json` that is the single source of
truth for what the model expects and what the directory contains. A reader never
has to guess a filename or restate the column list — it asks the manifest.

Note that `load_session` is TensorFlow-free: inspecting and validating a session
is cheap, and the model is only loaded when you actually score something.

In [1]:
from pathlib import Path

from mlpp.config import ColumnConfig
from mlpp.session import read_manifest, load_session

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SESSION_DIR = REPO_ROOT / "OUTPUTS" / "example"

# The column contract comes from the manifest, not from a hardcoded list.
features = read_manifest(SESSION_DIR).features
columns = ColumnConfig(
    input_columns=features.numeric_columns + features.categorical_columns,
    output_column=features.output_column,
    categorical_columns=features.categorical_columns,
)

session = load_session(SESSION_DIR, columns)
manifest = session.manifest

print(f"schema version : {manifest.schema_version}")
print(f"trained        : {manifest.created}")
print(f"target         : {features.output_column}")
print(f"inputs         : {len(features.numeric_columns)} numeric, "
      f"{len(features.categorical_columns)} categorical")
print(f"model features : {session.preprocessor.n_features} (after one-hot expansion)")
print(f"artifacts      : {len(manifest.artifacts)} files recorded")

schema version : 1
trained        : 2026-07-29T13:30:35
target         : target
inputs         : 13 numeric, 0 categorical
model features : 13 (after one-hot expansion)
artifacts      : 16 files recorded


## Baseline accuracy

Recorded at training time, one row per (train stage, test file) pair. The model
trains on each `TRAIN` file in turn while reusing the feature space fitted on the
first one, so later stages show the effect of additional data.

In [2]:
import pandas as pd

from mlpp.session import ROLE_METRICS

# Filename resolved through the manifest — session.py owns every name in here.
metrics_file = manifest.filenames_for(ROLE_METRICS)[0]
metrics = pd.read_csv(SESSION_DIR / metrics_file)
metrics.round(4)

,train_file,test_file,mse,rmse,mae,r2
0,sample_train_01.csv,sample_test_A.csv,0.0439,0.2096,0.1680,0.8967
1,sample_train_01.csv,sample_test_B.csv,0.0441,0.2100,0.1702,0.8968
2,sample_train_02.csv,sample_test_A.csv,0.0396,0.1989,0.1581,0.9070
3,sample_train_02.csv,sample_test_B.csv,0.0411,0.2028,0.1646,0.9038


## Scoring new data

The same session can score a CSV it has never seen, through the inference seam
(`mlpp.predict`). Predictions come back in the target's original units — the
target scaler is inverted on the way out, so these are directly comparable to the
truth column rather than living in standardised space.

Unseen categorical levels would be reported here rather than silently encoded as
zeros; this input has none.

In [3]:
from mlpp.data import read_csv_auto
from mlpp.predict import load_model, score_frame

frame = read_csv_auto(REPO_ROOT / "TEST" / "sample_test_A.csv")
scored = score_frame(session, load_model(session), frame)

comparison = pd.DataFrame({
    "actual": frame[features.output_column],
    "predicted": scored.predictions,
})
comparison["error"] = comparison["predicted"] - comparison["actual"]

print(f"rows scored     : {len(comparison)}")
print(f"unseen levels   : {scored.describe_unseen() or 'none'}")
print()
print(comparison.describe().loc[["mean", "std", "min", "max"]].round(4))
comparison.head()

rows scored     : 800
unseen levels   : none

      actual  predicted   error
mean  0.3992     0.4150  0.0158
std   0.6525     0.5933  0.1984
min  -1.5968    -1.4511 -0.6232
max   2.1111     2.0056  0.6081


,actual,predicted,error
0,0.092665,0.317122,0.224457
1,0.482640,0.393141,-0.089499
2,1.026389,0.838740,-0.187649
3,0.664019,0.482728,-0.181291
4,1.035573,0.982804,-0.052769


## Truth vs prediction

The interactive report below is a committed artifact of the reference run — pan
and zoom to inspect where the model tracks the signal and where it drifts.

In [4]:
from IPython.display import IFrame

from mlpp.session import ROLE_PREDICTION_ANALYSIS

# An IFrame src is resolved by the browser against this notebook's own directory
# (<repo>/notebooks), which is fixed — so the path is relative to that, never to
# the kernel's cwd. The filename itself comes from the manifest.
report_name = manifest.filenames_for(ROLE_PREDICTION_ANALYSIS)[0]
IFrame(f"../OUTPUTS/example/{report_name}", width="100%", height=850)

## Running it yourself

Everything above reads an existing session. To produce or consume one, from
`apps/backend/`:

```bash
# train a new session into OUTPUTS/<timestamp>/
uv run mlpp-train --epochs 5
uv run mlpp-train --help      # columns, loss, seed, batch size, plots…

# score a CSV against any session directory
uv run mlpp-predict --session ../../OUTPUTS/example \
  --input ../../TEST/sample_test_A.csv --output preds.csv
```

This notebook is generated from `scripts/build_notebook.py` and must not be
hand-edited; CI checks its source against the generator.